In [1]:
# Install Gemini API library
!pip install -U google-generativeai

  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.74.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.73.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.72.2-py3-none-any.whl.metadata (1.1 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.72.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached grpcio_status-1.71.2-py3-none-any


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports and Inititalizations

In [ ]:
import pandas as pd
import base64
import os
import google.generativeai as genai
from io import BytesIO
from PIL import Image
import nltk
import json
import itertools

env_path = ".env" 
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    print(f"Error: '{env_path}' file not found. Ensure it is in: {os.getcwd()}")
    API_KEYS = []

if API_KEYS:
    key_cycle = itertools.cycle(API_KEYS)
    print(f"Successfully loaded {len(API_KEYS)} API keys from .env for rotation.")
else:
    key_cycle = None
    print("Warning: No API keys loaded. The inference engine will fail.")

from doc_parsing_evaluator import ParsingEvaluator
nltk.download('punkt')

BASE_DIR = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "gemma-3-27b-it"
evaluator = ParsingEvaluator(group_name="OCR_Eval")

# --- GLOBAL EXPERIMENT CONFIG ---
resolutions = [None, 1024, 768] 

def process_image_for_tokens(image_bytes, max_edge=None):
    """Resizes image if needed and calculates approximate token counts."""
    img = Image.open(BytesIO(image_bytes))
    orig_w, orig_h = img.size
    
    orig_tokens = (orig_w // 28) * (orig_h // 28) 

    if max_edge and max(orig_w, orig_h) > max_edge:
        scale = max_edge / float(max(orig_w, orig_h))
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    new_w, new_h = img.size
    new_tokens = (new_w // 28) * (new_h // 28)
    
    token_savings_pct = round((1 - new_tokens/orig_tokens)*100, 2) if orig_tokens > 0 else 0
    
    stats = {
        "orig_res": f"{orig_w}x{orig_h}",
        "new_res": f"{new_w}x{new_h}",
        "orig_tokens": orig_tokens,
        "new_tokens": new_tokens,
        "token_savings_pct": token_savings_pct
    }
    return img, stats

Successfully loaded 9 API keys from .env for rotation.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\khann\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Inference Engine (Google cloud Models)

In [8]:
def query_gemini(img, task_type):
    """Sends a PIL image to Gemini API, rotating through available API keys"""
    current_key = next(key_cycle)
    genai.configure(api_key=current_key)
    
    if task_type == "table":
        prompt = "Parse this table into a clean, structured HTML format with <table> tags. Include all row and column spans."
    else:
        prompt = "Parse this document into LaTeX format. Provide only the LaTeX code."

    try:
        model = genai.GenerativeModel(model_name=MODEL_NAME)
        # Pass the PIL img directly
        response = model.generate_content([prompt, img])
        return response.text
    except Exception as e:
        print(f"Error with key {current_key[:10]}...: {e}")
        return ""

## Document Parsing Evaluation (LaTeX)

In [5]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

doc_results = {str(res): {} for res in resolutions}
doc_token_stats = {str(res): [] for res in resolutions}

for label, path in doc_tasks.items():
    if not os.path.exists(path): 
        print(f"⚠️ WARNING: Skipping {label}. Could not find file at:\n   {path}")
        continue
    
    df = pd.read_csv(path, sep='\t').head(5) 
    
    for res in resolutions:
        res_key = str(res)
        scores = []
        batch_savings = [] # Tracks savings just for this specific loop
        
        print(f"Processing {label} documents | Max Edge: {res if res else 'Original'}...")
        
        for _, row in df.iterrows():
            img_bytes = base64.b64decode(row['image'])
            ground_truth = str(row['answer']).strip() 
            
            img, stats = process_image_for_tokens(img_bytes, max_edge=res)
            doc_token_stats[res_key].append(stats['token_savings_pct'])
            batch_savings.append(stats['token_savings_pct'])
            
            prediction = query_gemini(img, "doc")
            score = evaluator.evaluate_single_doc_sample(ground_truth, prediction)
            scores.append(score)
            
        avg_score = sum(scores) / len(scores) if scores else 0
        doc_results[res_key][label] = avg_score
        avg_batch_savings = sum(batch_savings) / len(batch_savings) if batch_savings else 0
        
        print(f"  -> Accuracy: {avg_score:.4f} | Avg Token Savings: {avg_batch_savings:.1f}%\n")

Processing Scanned documents | Max Edge: Original...
  -> Accuracy: 0.5770 | Avg Token Savings: 0.0%

Processing Scanned documents | Max Edge: 1024...
  -> Accuracy: 0.5967 | Avg Token Savings: 31.6%

Processing Scanned documents | Max Edge: 768...
  -> Accuracy: 0.6240 | Avg Token Savings: 62.1%

Processing Photoed documents | Max Edge: Original...
  -> Accuracy: 0.4712 | Avg Token Savings: 0.0%

Processing Photoed documents | Max Edge: 1024...
  -> Accuracy: 0.5582 | Avg Token Savings: 66.3%

Processing Photoed documents | Max Edge: 768...
  -> Accuracy: 0.5142 | Avg Token Savings: 81.2%



## Table Parsing Evaluation (HTML)

In [9]:
table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results = {str(res): {} for res in resolutions}
table_token_stats = {str(res): [] for res in resolutions}

for label, path in table_tasks.items():
    if not os.path.exists(path): continue
        
    df = pd.read_csv(path, sep='\t').head(5) 
    
    for res in resolutions:
        res_key = str(res)
        scores = []
        batch_savings = []
        
        print(f"Processing {label} tables | Max Edge: {res if res else 'Original'}...")
        
        for _, row in df.iterrows():
            img_bytes = base64.b64decode(row['image'])
            ground_truth_html = str(row['answer']).strip() 
            
            img, stats = process_image_for_tokens(img_bytes, max_edge=res)
            table_token_stats[res_key].append(stats['token_savings_pct'])
            batch_savings.append(stats['token_savings_pct'])
            
            prediction = query_gemini(img, "table")
            score = evaluator.evaluate_single_table_sample(ground_truth_html, prediction)
            scores.append(score)
            
        avg_score = sum(scores) / len(scores) if scores else 0
        table_results[res_key][label] = avg_score
        avg_batch_savings = sum(batch_savings) / len(batch_savings) if batch_savings else 0
        
        print(f"  -> Accuracy: {avg_score:.4f} | Avg Token Savings: {avg_batch_savings:.1f}%\n")

Processing Scanned tables | Max Edge: Original...
  -> Accuracy: 0.7121 | Avg Token Savings: 0.0%

Processing Scanned tables | Max Edge: 1024...
  -> Accuracy: 0.7112 | Avg Token Savings: 18.8%

Processing Scanned tables | Max Edge: 768...
  -> Accuracy: 0.7122 | Avg Token Savings: 28.3%

Processing Photoed tables | Max Edge: Original...
  -> Accuracy: 0.5952 | Avg Token Savings: 0.0%

Processing Photoed tables | Max Edge: 1024...
  -> Accuracy: 0.6774 | Avg Token Savings: 56.5%

Processing Photoed tables | Max Edge: 768...
  -> Accuracy: 0.6989 | Avg Token Savings: 75.9%



## Final Report 

In [10]:
# Helper to safely calculate combined average without division-by-zero errors
def get_total_avg(res_key):
    total_list = doc_token_stats[res_key] + table_token_stats[res_key]
    return sum(total_list) / max(1, len(total_list))

summary = {
    "Experiment": "Resolution vs Accuracy Tradeoff (5 Images)",
    "Model": MODEL_NAME,
    "Document Parsing (NED-LaTeX)": doc_results,
    "Table Parsing (HTML-Sim)": table_results,
    "Average Token Savings": {
        "None (Original)": get_total_avg("None"),
        "1024": get_total_avg("1024"),
        "768": get_total_avg("768")
    }
}

report_path = os.path.join(OUT_DIR, "resolution_tradeoff_report.json")
with open(report_path, "w") as f:
    json.dump(summary, f, indent=4)

print(f"Evaluation Complete. Trade-off report saved to:\n{report_path}")

Evaluation Complete. Trade-off report saved to:
D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\resolution_tradeoff_report.json
